# Ground truth vs COLMAP vs MASt3R — camera pose estimation only

Input: a zip of your `Plant1` capture folder (`frame_XXXX.png` + `frame_XXXX.yaml`,
where each yaml has the robot-FK ground-truth `camera_pose`).

This notebook estimates **camera poses only** — no dense reconstruction, no
point clouds, no meshes:
1. Unzips your dataset in `/content`
2. Runs **COLMAP** sparse SfM (`feature_extractor` -> `matcher` -> `mapper`) to get poses
3. Runs **MASt3R**'s `sparse_global_alignment` to get poses (this is MASt3R's
   pose-estimation step; the dense point-cloud export from your original
   notebook is skipped entirely)
4. Converts ground truth, COLMAP, and MASt3R poses into one common
   `frame_name -> 4x4 camera-to-world matrix` format
5. Saves the pose files to a Drive folder you choose

The COLMAP and MASt3R install cells are copied as-is from your existing
`colmap_colab_dense_with_metrics.ipynb` / `MASt3R_Plant_Final.ipynb` notebooks.

**Runtime:** GPU (Runtime -> Change runtime type -> T4 GPU) before running anything below.

## 0. Check GPU

In [1]:
!nvidia-smi

Sat Jul 25 15:43:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Mount Google Drive + choose save paths

Set `DRIVE_ROOT` to wherever you want everything for this run to land.
Sub-folders for COLMAP / MASt3R / ground truth / comparison are created under it.

In [12]:


import os

# <<< EDIT THIS to the Drive folder you want results saved under >>>
DRIVE_ROOT = "/content/drive/MyDrive/jetcobot_colab/notebooks/pose"

COLMAP_DRIVE_DIR   = os.path.join(DRIVE_ROOT, "colmap")
MAST3R_DRIVE_DIR   = os.path.join(DRIVE_ROOT, "mast3r")
GT_DRIVE_DIR       = os.path.join(DRIVE_ROOT, "ground_truth")
COMPARE_DRIVE_DIR  = os.path.join(DRIVE_ROOT, "comparison")

for d in [COLMAP_DRIVE_DIR, MAST3R_DRIVE_DIR, GT_DRIVE_DIR, COMPARE_DRIVE_DIR]:
    os.makedirs(d, exist_ok=True)

print("Results will be saved under:", DRIVE_ROOT)

Results will be saved under: /content/drive/MyDrive/jetcobot_colab/notebooks/pose


## 2. Upload and unzip your dataset

Upload the zip of your `Plant1` folder (350 `frame_XXXX.png` + 350 `frame_XXXX.yaml`)
when prompted.

In [8]:
# <<< EDIT THIS to the full path of your zip file in Google Drive >>>
ZIP_FILE_PATH_IN_DRIVE = "/content/drive/MyDrive/jetcobot_colab/notebooks/pose/Plant1.zip" # Replace with your actual path

In [9]:
import zipfile

zip_name = ZIP_FILE_PATH_IN_DRIVE

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/raw_dataset')

print("Extracted to /content/raw_dataset")

Extracted to /content/raw_dataset


In [2]:
import os, glob

# find the folder that actually contains the images + yaml, regardless of
# exact zip layout (flat folder vs. nested Plant1/ subfolder etc.)
candidates = [r for r, d, f in os.walk('/content/raw_dataset')
              if any(fn.lower().endswith('.png') for fn in f)
              and any(fn.lower().endswith('.yaml') for fn in f)]
assert candidates, "No matching .png + .yaml pair found after extracting the zip — check the zip contents"
DATASET_ROOT = candidates[0]

image_paths = sorted(glob.glob(os.path.join(DATASET_ROOT, '*.png')))
yaml_paths  = sorted(glob.glob(os.path.join(DATASET_ROOT, '*.yaml')))

print(f"Dataset root: {DATASET_ROOT}")
print(f"Found {len(image_paths)} images, {len(yaml_paths)} yaml files")
assert len(image_paths) == len(yaml_paths), "Image / yaml count mismatch — check the dataset"

Dataset root: /content/raw_dataset/Plant1
Found 351 images, 351 yaml files


## 3. Install COLMAP

Copied as-is from `colmap_colab_dense_with_metrics.ipynb`.

In [11]:
!apt-get update -qq
!apt-get install -y colmap

!pip install -q condacolab
import condacolab
condacolab.install()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libamd2 libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libcamd2 libccolamd2 libceres2
  libcholmod3 libcolamd2 libcxsparse3 libdouble-conversion3 libevdev2
  libfreeimage3 libgflags2.2 libglew2.2 libgoogle-glog0v5 libgtk-3-0
  libgtk-3-bin libgtk-3-common libgudev-1.0-0 libilmbase25 libinput-bin
  libinput10 libjxr0 libmd4c0 libmetis5 libmtdev1 libopenexr25 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libraw20
  librsvg2-common libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common
  libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0
  l

In [12]:
!rm -f /usr/local/conda-meta/pinned
!cat /usr/local/conda-meta/pinned 2>/dev/null || echo "pin file removed"

!mamba install -y -c conda-forge colmap "python=3.11"

!mamba remove -y colmap faiss faiss-gpu 2>/dev/null
!mamba install -y -c conda-forge colmap

!mamba create -y -n colmap_env -c conda-forge colmap

!mamba install -y -n colmap_env -c conda-forge faiss

!mamba env remove -y -n colmap_env
!mamba create -y -n colmap_env -c conda-forge python=3.11 colmap faiss

!mamba env remove -y -n colmap_env
!mamba create -y -n colmap_env -c conda-forge python=3.11 colmap faiss openimageio

pin file removed

Looking for: ['colmap', 'python=3.11']

[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   8%
conda-forge/noarch    23%[+] 0.3s
conda-forge/linux-64  16%
conda-forge/noarch    40%[+] 0.4s
conda-forge/linux-64  22%
conda-forge/noarch    52%[+] 0.5s
conda-forge/linux-64  28%
conda-forge/noarch    63%[+] 0.6s
conda-forge/linux-64  35%
conda-forge/noarch    77%[+] 0.7s
conda-forge/linux-64  42%
conda-forge/noarch    92%[+] 0.8s
conda-forge/linux-64  45%
conda-forge/noarch    99%conda-forge/noarch                                
[+] 0.9s
conda-forge/linux-64  45%[+] 1.0s
conda-forge/linux-64  49%[+] 1.1s
conda-forge/linux-64  56%[+] 1.2s
conda-forge/linux-64  65%[+] 1.3s
conda-forge/linux-64  72%[+] 1.4s
conda-forge/linux-64  79%[+] 1.5s
conda-forge/linux-64  86%[+] 1.6s
conda-forge/linux-64  92%[+] 1.7s
conda-forge/linux-64  99%conda-forge/linux-64                              
Transaction

  Prefix: /usr/local

  Updating

In [13]:
!mamba run -n colmap_env colmap patch_match_stereo --help | grep -i cuda

!mamba run -n colmap_env ldd $(mamba run -n colmap_env which colmap) | grep "not found"

I20260725 15:56:53.496312 138881184587776 option_manager.cc:1214] COLMAP 4.1.1 (Commit Unknown on Unknown with CUDA)
I20260725 15:56:53.496563 138881184587776 option_manager.cc:1216] Options can either be specified via command-line or by defining them in a .ini project file passed to `--project_path`.
  -h [ --help ] 
  --project_path arg
  --default_random_seed arg (=0)
  --log_target arg (=stderr_and_file)   {stderr, stdout, file, stderr_and_file}
  --log_path arg
  --log_level arg (=0)
  --log_severity arg (=0)               0:INFO, 1:WARNING, 2:ERROR, 3:FATAL
  --log_color arg (=1)
  --workspace_path arg                  Path to the folder containing the 
                                        undistorted images
  --workspace_format arg (=COLMAP)      {COLMAP, PMVS}
  --pmvs_option_name arg (=option-all)
  --config_path arg
  --PatchMatchStereo.max_image_size arg (=-1)
  --PatchMatchStereo.gpu_index arg (=-1)
  --PatchMatchStereo.depth_min arg (=-1)
  --PatchMatchStereo.depth_max 

## 4. Run COLMAP — sparse SfM only (no dense stage)

Your existing COLMAP notebook started from an already-built sparse model and
ran the GPU dense stage (`image_undistorter` -> `patch_match_stereo` ->
`stereo_fusion`) on top. Since you only want poses, we stop after `mapper` —
that's COLMAP's sparse SfM step and it's exactly what produces the camera
poses. No dense stage, no point cloud.

In [3]:
# Subsample: take every other frame (0, 2, 4, ...) to halve the dataset
image_paths = image_paths[::2]
yaml_paths  = yaml_paths[::2]

print(f"Subsampled to {len(image_paths)} images and {len(yaml_paths)} yaml files")
assert len(image_paths) == len(yaml_paths), "Image / yaml count mismatch after subsampling"

Subsampled to 176 images and 176 yaml files


In [4]:
import os

COLMAP_WS = "/content/colmap_ws"
os.makedirs(os.path.join(COLMAP_WS, "images"), exist_ok=True)

# COLMAP just needs the images in their own folder
import shutil
for p in image_paths:
    shutil.copy2(p, os.path.join(COLMAP_WS, "images", os.path.basename(p)))

print(f"Copied {len(image_paths)} images to {COLMAP_WS}/images")

Copied 176 images to /content/colmap_ws/images


In [6]:
!rm -f {DB_PATH}

In [19]:
import subprocess

# ---- EDIT THESE to your actual simulated camera intrinsics ----
CAMERA_MODEL  = "PINHOLE"     # known intrinsics -> tell COLMAP the real fx, fy, cx, cy
                               # instead of letting it self-calibrate (was SIMPLE_RADIAL,
                               # which guessed and produced 0 confidently-calibrated pairs)
SINGLE_CAMERA = True
FX, FY, CX, CY = 379.99, 379.99, 320.0, 240.0
# -----------------------------------------------------------------

!rm -f {DB_PATH}

subprocess.run([
    "mamba", "run", "-n", "colmap_env", "colmap", "feature_extractor",
    "--database_path", DB_PATH,
    "--image_path", IMAGES_PATH,
    "--ImageReader.single_camera", "1" if SINGLE_CAMERA else "0",
    "--ImageReader.camera_model", CAMERA_MODEL,
    "--ImageReader.camera_params", f"{FX},{FY},{CX},{CY}",
], check=True)

subprocess.run([
    "mamba", "run", "-n", "colmap_env", "colmap", "exhaustive_matcher",
    "--database_path", DB_PATH,
], check=True)

subprocess.run([
    "mamba", "run", "-n", "colmap_env", "colmap", "mapper",
    "--database_path", DB_PATH,
    "--image_path", IMAGES_PATH,
    "--output_path", SPARSE_PATH,
], check=True)

models = sorted(d for d in os.listdir(SPARSE_PATH) if os.path.isdir(os.path.join(SPARSE_PATH, d)))
print("Reconstructed sub-models:", models)
assert models, "mapper produced no reconstruction — intrinsics likely still wrong, check FX/FY/CX/CY"

Reconstructed sub-models: ['0', '1', 'sparse_txt']


In [21]:
import subprocess, re

def analyze_model_full(path):
    result = subprocess.run(
        ["mamba", "run", "-n", "colmap_env", "colmap", "model_analyzer", "--path", path],
        check=True, capture_output=True, text=True,
    )
    log = result.stderr
    def grab(pattern, cast=float):
        m = re.search(pattern, log)
        return cast(m.group(1)) if m else None
    return {
        "registered_images": grab(r"Registered images:\s*(\d+)", int),
        "points": grab(r"Points:\s*(\d+)", int),
        "mean_track_length": grab(r"Mean track length:\s*([\d.]+)"),
        "mean_reproj_error_px": grab(r"Mean reprojection error:\s*([\d.]+)px"),
    }

report = analyze_model_full(os.path.join(SPARSE_PATH, "0"))
print(report)

{'registered_images': 176, 'points': 14034, 'mean_track_length': 9.312527, 'mean_reproj_error_px': 0.576982}


## 5. Export COLMAP poses to the common format

Converts COLMAP's `images.txt` (quaternion + translation, world-to-camera)
into `{frame_name: 4x4 camera-to-world matrix}`, the same format used for
MASt3R and ground truth below.

In [22]:
os.makedirs(os.path.join(SPARSE_PATH, "sparse_txt"), exist_ok=True)

!mamba run -n colmap_env colmap model_converter \
    --input_path {SPARSE_PATH}/0 \
    --output_path {SPARSE_PATH}/sparse_txt \
    --output_type TXT

In [23]:
import numpy as np

def qt_to_c2w(qw, qx, qy, qz, tx, ty, tz):
    R = np.array([
        [1 - 2*qy**2 - 2*qz**2, 2*qx*qy - 2*qz*qw,     2*qx*qz + 2*qy*qw],
        [2*qx*qy + 2*qz*qw,     1 - 2*qx**2 - 2*qz**2, 2*qy*qz - 2*qx*qw],
        [2*qx*qz - 2*qy*qw,     2*qy*qz + 2*qx*qw,     1 - 2*qx**2 - 2*qy**2],
    ])
    t = np.array([tx, ty, tz])
    c2w = np.eye(4)
    c2w[:3, :3] = R.T
    c2w[:3, 3] = -R.T @ t
    return c2w

colmap_poses = {}
images_txt = os.path.join(SPARSE_PATH, "sparse_txt", "images.txt")
with open(images_txt) as f:
    lines = [l for l in f if l.strip() and not l.startswith("#")]
for line in lines[::2]:  # each image occupies 2 lines; the pose is on the first
    parts = line.split()
    qw, qx, qy, qz, tx, ty, tz = map(float, parts[1:8])
    name = parts[9]
    colmap_poses[name] = qt_to_c2w(qw, qx, qy, qz, tx, ty, tz)

print(f"Recovered poses for {len(colmap_poses)} / {len(image_paths)} images (COLMAP drops unregistered images)")

np.savez(os.path.join(COLMAP_WS, "colmap_poses_c2w.npz"), **colmap_poses)

Recovered poses for 176 / 176 images (COLMAP drops unregistered images)


In [24]:
import numpy as np
d = np.load(os.path.join(COLMAP_WS, "colmap_poses_c2w.npz"))
print(f"{len(d.files)} poses saved")

176 poses saved


In [25]:
import shutil

os.makedirs(COLMAP_DRIVE_DIR, exist_ok=True)

shutil.copy2(os.path.join(COLMAP_WS, "colmap_poses_c2w.npz"), COLMAP_DRIVE_DIR)

!cp -r {SPARSE_PATH}/sparse_txt "{COLMAP_DRIVE_DIR}/sparse_txt"

print("Saved COLMAP poses to", COLMAP_DRIVE_DIR)

Saved COLMAP poses to /content/drive/MyDrive/jetcobot_colab/notebooks/pose/colmap


In [26]:
import sqlite3
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM keypoints")
print("images with keypoints:", cur.fetchone()[0])
cur.execute("SELECT rows FROM keypoints")
counts = [r[0] for r in cur.fetchall()]
print("keypoints per image — min/median/max:", min(counts), sorted(counts)[len(counts)//2], max(counts))
cur.execute("SELECT COUNT(*) FROM matches")
print("image pairs with matches:", cur.fetchone()[0])
conn.close()

images with keypoints: 176
keypoints per image — min/median/max: 334 992 2482
image pairs with matches: 15400


In [27]:
import sqlite3
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM two_view_geometries")
print("pairs with verified geometry:", cur.fetchone()[0])

cur.execute("SELECT rows FROM two_view_geometries WHERE rows > 0")
verified_counts = [r[0] for r in cur.fetchall()]
if verified_counts:
    print("verified inlier count — min/median/max:",
          min(verified_counts), sorted(verified_counts)[len(verified_counts)//2], max(verified_counts))
else:
    print("no pairs passed geometric verification at all")
conn.close()

pairs with verified geometry: 15400
verified inlier count — min/median/max: 15 80 1248


In [28]:
import sqlite3
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT rows, config FROM two_view_geometries")
rows = cur.fetchall()
verified = [(r, c) for r, c in rows if r > 0]
print("verified pairs (rows>0):", len(verified), "/", len(rows))

from collections import Counter
config_counts = Counter(c for r, c in verified)
print("config type counts:", config_counts)
conn.close()

verified pairs (rows>0): 5467 / 15400
config type counts: Counter({2: 4428, 6: 515, 3: 509, 7: 15})


## 6. Install MASt3R

Copied as-is from `MASt3R_Plant_Final.ipynb`.

In [29]:
%cd /content
!git clone --recursive https://github.com/naver/mast3r
%cd /content/mast3r
!pip install -q -r requirements.txt
!pip install -q -r dust3r/requirements.txt
!pip install -q -r dust3r/requirements_optional.txt || true
!pip install -q open3d

!mkdir -p checkpoints
!wget -q --show-progress -O checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth \
    https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth

/content
Cloning into 'mast3r'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (61/61), done.
Receiving objects: 100% (269/269), 3.59 MiB | 40.41 MiB/s, done.
remote: Total 269 (delta 115), reused 109 (delta 109), pack-reused 99 (from 1)
Resolving deltas: 100% (151/151), done.
Submodule 'dust3r' (https://github.com/naver/dust3r) registered for path 'dust3r'
Cloning into '/content/mast3r/dust3r'...
remote: Enumerating objects: 611, done.        
remote: Total 611 (delta 0), reused 0 (delta 0), pack-reused 611 (from 1)        
Receiving objects: 100% (611/611), 756.60 KiB | 1.25 MiB/s, done.
Resolving deltas: 100% (355/355), done.
Submodule path 'dust3r': checked out '3cc8c88c413bb9e34c41db0e0eef99c2ee010b12'
Submodule 'croco' (https://github.com/naver/croco) registered for path 'dust3r/croco'
Cloning into '/content/mast3r/dust3r/croco'...
remote: Enumerating objects: 198, done.        
remote: Counting objects:

## 7. Run MASt3R on the same images

Same `sparse_global_alignment` call as `MASt3R_Plant_Final.ipynb`, pointed at
`image_paths` from the unzipped dataset instead of a separate upload.

In [34]:
!pip install -q roma

In [39]:
import sys
!{sys.executable} -m pip install roma

  Using cached roma-1.5.6-py3-none-any.whl.metadata (5.5 kB)
Using cached roma-1.5.6-py3-none-any.whl (25 kB)


In [40]:
import sys
sys.path.insert(0, '/content/mast3r')
sys.path.insert(0, '/content/mast3r/dust3r')

from mast3r.model import AsymmetricMASt3R
from mast3r.cloud_opt.sparse_ga import sparse_global_alignment
from mast3r.image_pairs import make_pairs
from dust3r.utils.image import load_images
import tempfile

device = "cuda"

model = AsymmetricMASt3R.from_pretrained(
    "naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric"
).to(device)

imgs = load_images(image_paths, size=512)
print(f"Loaded {len(imgs)} images for inference")

pairs = make_pairs(imgs, scene_graph='logwin-8-cyclic', prefilter=None, symmetrize=True)
print(f"{len(pairs)} pairs from {len(imgs)} images")

cache_dir = tempfile.mkdtemp(prefix='mast3r_cache_')

scene = sparse_global_alignment(
    image_paths,
    pairs,
    cache_dir,
    model,
    lr1=0.07, niter1=500,
    lr2=0.014, niter2=200,
    device=device,
    opt_depth=False,
    matching_conf_thr=5.0,
)
print("Alignment done.")

/!\ module trimesh is not installed, cannot visualize results /!\


/content/mast3r/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.75GB            

model.safetensors: downloading bytes:           |  0.00B            

...

init focals = [551.0157  554.5068  506.2433  584.28534 588.96826 564.2212  571.6666
 542.3483  513.64105 529.3962  540.69183 562.34906 626.76154 567.9367
 556.2403  577.81616 570.7308  567.6121  566.634   555.959   521.208
 512.285   509.65918 609.06024 503.42975 487.1483  512.3686  538.96747
 544.0942  557.7651  565.2817  537.2627  538.7889  524.2757  585.5251
 533.2985  547.1006  516.50085 553.56415 558.1878  550.9264  549.91876
 547.9804  564.56995 561.4689  635.51996 528.3146  555.09564 530.7153
 475.65573 485.88696 501.10513 546.10516 526.0291  524.72314 519.48114
 654.18945 481.08157 491.7781  477.40924 490.7032  517.5864  606.4466
 491.54367 505.52155 501.28488 497.66495 504.2908  511.01657 548.1216
 517.47955 512.21783 495.46313 572.1005  571.80896 489.76602 529.50397
 539.10986 545.7931  548.1652  537.3787  525.0793  507.182   522.7555
 590.00824 523.2899  529.4378  514.54785 539.4506  536.2573  546.3399
 537.6399  534.33325 552.14215 536.77405 596.5974  531.2984  543.848
 515

  0%|          | 0/500 [00:00<?, ?it/s]/content/mast3r/mast3r/cloud_opt/sparse_ga.py:457: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  loss = float(loss)
100%|██████████| 500/500 [16:19<00:00,  1.96s/it, lr=0.0000, loss=0.266]


>> final loss = 0.2658240497112274


100%|██████████| 200/200 [07:42<00:00,  2.31s/it, lr=0.0000, loss=1.309]


>> final loss = 1.308645248413086
Final focals = [ 822.7856   810.04065  775.78235  846.34753  854.50195  850.5563
  793.40735  764.3681   868.81433  856.8338   862.7403  1113.2188
  481.95615 1217.8279   793.55225  772.74164  793.6124   860.97925
  877.2538   862.9839   802.35034  766.58813  779.1174   581.5478
  782.70544  787.0521   767.74225  728.0628   755.71844  767.41943
  796.0373   783.82965  781.60223  782.8861   644.06287  780.4414
  800.0481   787.90485  799.52295  796.28107  794.8001   800.23584
  798.7624   801.66345  800.9429   510.05206  787.1103   792.0835
  768.48505  764.6299   764.6638   765.01025  762.2062   759.1123
  762.27893  763.274    509.41113  762.9873   753.7995   764.5584
  763.2533   787.6851   524.4975   783.4433   780.73584  758.3944
  732.55023  709.7527   715.8499   719.2633   753.25616  726.3149
  699.6709   502.23422  731.041    752.26447  731.5672   734.8257
  742.96796  754.76514  730.3574   749.3809   754.03424  745.5394
  812.30457  762.29376  

## 8. Export MASt3R poses to the common format

`scene.get_im_poses()` is already camera-to-world, so this is a straight
save keyed by filename (same `{frame_name: 4x4}` format as COLMAP/ground truth).
The dense point-cloud export (`get_dense_pts3d` / Open3D cleanup / `.ply`) from
your original notebook is skipped — poses only.

In [41]:
poses_c2w = scene.get_im_poses().detach().cpu().numpy()   # (N,4,4)
focals = scene.get_focals().detach().cpu().numpy().reshape(-1)
pps = scene.get_principal_points().detach().cpu().numpy()

N = len(focals)
intrinsics = np.zeros((N, 3, 3), dtype=np.float32)
for i in range(N):
    intrinsics[i, 0, 0] = focals[i]
    intrinsics[i, 1, 1] = focals[i]
    intrinsics[i, 0, 2] = pps[i, 0]
    intrinsics[i, 1, 2] = pps[i, 1]
    intrinsics[i, 2, 2] = 1.0

mast3r_poses = {os.path.basename(p): poses_c2w[i] for i, p in enumerate(image_paths)}
np.savez("/content/mast3r_poses_c2w.npz", **mast3r_poses)
np.save("/content/mast3r_intrinsics.npy", intrinsics)

print(f"Saved poses for {len(mast3r_poses)} images")

Saved poses for 176 images


In [42]:
os.makedirs(MAST3R_DRIVE_DIR, exist_ok=True)

for fname in ["/content/mast3r_poses_c2w.npz", "/content/mast3r_intrinsics.npy"]:
    shutil.copy2(fname, MAST3R_DRIVE_DIR)

print("Saved MASt3R poses to", MAST3R_DRIVE_DIR)

Saved MASt3R poses to /content/drive/MyDrive/jetcobot_colab/notebooks/pose/mast3r


## 9. Ground-truth poses (from the yaml, same common format)

Parses `camera_pose.position` + `camera_pose.orientation_quaternion` from
each `frame_XXXX.yaml` into the same `{frame_name: 4x4 c2w}` format as
COLMAP/MASt3R, so the three are directly comparable frame-by-frame.

In [43]:
import yaml as pyyaml

def quat_to_R(x, y, z, w):
    n = np.sqrt(x*x + y*y + z*z + w*w)
    if n < 1e-8:
        return np.eye(3)
    x, y, z, w = x/n, y/n, z/n, w/n
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w),     2*(x*z + y*w)],
        [2*(x*y + z*w),     1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w),     2*(y*z + x*w),     1 - 2*(x*x + y*y)],
    ])

gt_poses = {}
for yp in yaml_paths:
    with open(yp) as f:
        data = pyyaml.safe_load(f)
    cam = data.get("camera_pose", {})
    pos = cam.get("position", {})
    quat = cam.get("orientation_quaternion", {})

    R = quat_to_R(quat.get("x", 0.0), quat.get("y", 0.0),
                  quat.get("z", 0.0), quat.get("w", 1.0))
    t = np.array([pos.get("x", 0.0), pos.get("y", 0.0), pos.get("z", 0.0)])

    c2w = np.eye(4)
    c2w[:3, :3] = R
    c2w[:3, 3] = t

    # match COLMAP/MASt3R keys, which are by image filename
    img_name = os.path.splitext(os.path.basename(yp))[0] + ".png"
    gt_poses[img_name] = c2w

print(f"Parsed ground-truth poses for {len(gt_poses)} frames")
np.savez("/content/gt_poses_c2w.npz", **gt_poses)

shutil.copy2("/content/gt_poses_c2w.npz", GT_DRIVE_DIR)
print("Saved ground truth poses to", GT_DRIVE_DIR)

Parsed ground-truth poses for 176 frames
Saved ground truth poses to /content/drive/MyDrive/jetcobot_colab/notebooks/pose/ground_truth


## 10. Done

Everything is saved under `DRIVE_ROOT`:

- `colmap/colmap_poses_c2w.npz`, `colmap/sparse_txt/`
- `mast3r/mast3r_poses_c2w.npz`, `mast3r/mast3r_intrinsics.npy`
- `ground_truth/gt_poses_c2w.npz`

No point clouds or meshes — poses only. All three `*_poses_c2w.npz` files
share the same format — an `.npz` archive where each key is the image
filename (e.g. `frame_0123.png`) and each value is a 4x4 camera-to-world
matrix — so `comparison/` is ready for you to load all three and compute
alignment/error metrics (e.g. with `scipy`'s Umeyama / Procrustes alignment)
in a follow-up notebook.

In [44]:
print("DRIVE_ROOT contents:")
for root, dirs, fs in os.walk(DRIVE_ROOT):
    level = root.replace(DRIVE_ROOT, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in fs:
        print("  " * (level + 1) + f)

DRIVE_ROOT contents:
pose/
  Plant1.zip
  colmap/
    colmap_poses_c2w.npz
    sparse_txt/
      rigs.txt
      cameras.txt
      frames.txt
      images.txt
      points3D.txt
  mast3r/
    mast3r_poses_c2w.npz
    mast3r_intrinsics.npy
  ground_truth/
    gt_poses_c2w.npz
  comparison/
